# Getting started

`ebolasim` is a Python interface to the EbolaSim C model. The
workflow is deliberately small: create a parameter file using the
exact names accepted by the C model, create a `Sim`, run it, and
read or plot the outputs.

The examples below use a synthetic demo population. Release wheels
for Linux x86_64 can include the compiled executable. Source
checkouts without a bundled executable can still create parameters
and cluster commands.


In [ ]:
import ebolasim as es

pars = es.demo_pars().set({
    "Population size": 2400,
    "Number of realisations": 20,
    "Sampling time": 90,
    "Reproduction number": 1.6,
    "Initial number of infecteds": 1,
    "Output incidence by administrative unit": 1,
    "Output age file": 1,
})

{
    "parameters": len(pars.raw),
    "population": pars.raw["Population size"],
    "realisations": pars.raw["Number of realisations"],
    "days": pars.raw["Sampling time"],
    "r0": pars.raw["Reproduction number"],
}


The object is still a faithful C-model parameter file. Writing it
creates the bracketed text format read by `ReadParams()` upstream.


In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = pars.write(Path(tmp) / "parameters.txt")
    print(path.read_text(encoding="utf-8").splitlines()[:12])


A small smoke run is useful for local checks and CI. If the current
environment does not have a bundled executable, this cell still
shows the exact command that would be submitted to a cluster.


In [ ]:
smoke_pars = es.demo_pars({
    "Population size": 24,
    "Number of realisations": 1,
    "Sampling time": 7,
    "Reproduction number": 1.6,
})

with tempfile.TemporaryDirectory() as tmp:
    smoke = es.Sim(
        smoke_pars,
        label="smoke",
        outdir=Path(tmp) / "notebook-smoke",
        threads=1,
    )
    executable = es.resolve_executable(required=False)

    if executable is None:
        plan = smoke.command()
        print("No bundled executable is installed in this environment.")
        print(plan.shell_command)
        summary = None
    else:
        smoke.run(timeout=30)
        summary = smoke.summary

summary


In [ ]:
if smoke.results is None:
    print("Plot skipped because the model executable was not available.")
else:
    smoke.plot(y="I")


Scenario comparisons use ordinary parameter copies. The important
habit is that all names remain exact C-model names.


In [ ]:
higher_r0_pars = pars.copy().set({
    "Reproduction number": 2.0,
    "Include contact tracing": 1,
})

with tempfile.TemporaryDirectory() as tmp:
    run_root = Path(tmp)
    baseline = es.Sim(pars, label="baseline", outdir=run_root / "baseline", threads=4)
    higher_r0 = es.Sim(
        higher_r0_pars,
        label="higher_r0",
        outdir=run_root / "higher_r0",
        threads=4,
    )
    commands = [baseline.command().shell_command, higher_r0.command().shell_command]

commands
